# 🌐 Projeto: Ingestão por Web Scraping e Visualização de Dados (UFRN / EAJ)
**Disciplina:** Ingestão e Análise de Dados / Web Scraping — **UFRN**  
**Objetivo:**
1. Coletar os links de notícias (URLs) da página de notícias da UFRN com a keyword `EAJ`: `https://www.ufrn.br/imprensa/noticias/filtros?keyword=EAJ` e salvar em arquivo apropriado (`noticias_eaj.txt`) contendo o ano de publicação e a URL correspondente.
2. Construir um gráfico de visualização (barras/pizza) mostrando a quantidade de notícias publicadas que citam a EAJ por ano.
3. Apresentar os resultados em dashboard interativo no Streamlit.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

## 📦 1. Instalação das Dependências no Ambiente Colab

In [ ]:
!pip install requests pandas plotly matplotlib streamlit -q

## 🕷️ 2. Implementação do Web Scraping (Requisito 1)
A função abaixo realiza a coleta paginada de todas as notícias do portal da UFRN associadas à palavra-chave **EAJ**.

In [ ]:
import os
import json
import requests
import urllib3
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# Desativar avisos de SSL
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

BASE_PORTAL_URL = "https://www.ufrn.br/imprensa/noticias/"
API_BUSCA_URL = "https://webcache01-producao.info.ufrn.br/admin/portal-ufrn/wp-json/wp/v2/noticias-busca/"
KEYWORD = "EAJ"

def fetch_eaj_news(keyword="EAJ", per_page=100):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }
    all_news = []
    current_page = 1
    total_pages = 1
    
    print(f"[*] Iniciando coleta de notícias da UFRN (Keyword: {keyword})...")
    
    while True:
        params = {
            "_embed": "",
            "per_page": per_page,
            "page": current_page,
            "tags": keyword
        }
        try:
            response = requests.get(API_BUSCA_URL, params=params, headers=headers, verify=False, timeout=20)
            if response.status_code != 200:
                break
                
            header_pages = response.headers.get("X-WP-TotalPages")
            if header_pages:
                total_pages = int(header_pages)
                
            items = response.json()
            if not isinstance(items, list) or len(items) == 0:
                break
                
            for item in items:
                news_id = item.get("id")
                slug = item.get("slug", "")
                date_str = item.get("date", "")
                year = None
                
                if date_str:
                    try:
                        dt = datetime.fromisoformat(date_str)
                        year = dt.year
                    except Exception:
                        year = int(date_str[:4]) if len(date_str) >= 4 else None
                        
                title = item.get("title", {}).get("rendered", "")
                url = f"{BASE_PORTAL_URL}{news_id}/{slug}" if slug else f"{BASE_PORTAL_URL}{news_id}"
                
                all_news.append({
                    "ano": year,
                    "url": url,
                    "titulo": title,
                    "id": news_id
                })
                
            print(f" -> Página {current_page}/{total_pages} processada | Total de notícias: {len(all_news)}")
            
            if current_page >= total_pages:
                break
            current_page += 1
            
        except Exception as e:
            print(f"[Erro] {e}")
            break
            
    print(f"[+] Coleta concluída com {len(all_news)} notícias!")
    return all_news

# Executa a coleta
news_data = fetch_eaj_news(KEYWORD)

## 💾 3. Salvando os Dados em Arquivo .TXT, .CSV e .JSON (Requisito 1)
Gravando o arquivo `noticias_eaj.txt` contendo o **Ano** e a **URL** de cada notícia.

In [ ]:
# 1. Salvar em arquivo .txt
txt_filename = "noticias_eaj.txt"
with open(txt_filename, "w", encoding="utf-8") as f:
    f.write("# ANO\tURL\tTITULO\n")
    for item in news_data:
        f.write(f"{item['ano']}\t{item['url']}\t{item['titulo']}\n")
print(f"✅ Arquivo TXT gerado com sucesso: {txt_filename}")

# 2. Salvar em CSV para análise de dados
df_news = pd.DataFrame(news_data)
df_news.to_csv("noticias_eaj.csv", index=False, encoding="utf-8")
print("✅ Arquivo CSV gerado com sucesso: noticias_eaj.csv")

# 3. Exibir as 10 primeiras linhas do arquivo TXT
print("\n--- Visualização das primeiras linhas do arquivo TXT ---")
with open(txt_filename, "r", encoding="utf-8") as f:
    for _ in range(10):
        print(f.readline().strip())

## 📊 4. Construção dos Gráficos de Visualização por Ano (Requisito 2)
Agrupamento e contagem do número de notícias publicadas por ano que citam a **EAJ**.

In [ ]:
# Agrupamento por ano
df_valid = df_news.dropna(subset=["ano"]).copy()
df_valid["ano"] = df_valid["ano"].astype(int)
contagem_ano = df_valid["ano"].value_counts().sort_index()

print("--- Contagem de Notícias por Ano ---")
print(contagem_ano)

# Gráfico com Matplotlib / Seaborn Style
plt.figure(figsize=(10, 5))
bars = plt.bar(contagem_ano.index.astype(str), contagem_ano.values, color='#0284c7', edgecolor='#0369a1', width=0.6)
plt.title('Quantidade de Notícias Publicadas que citam a EAJ por Ano (UFRN)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Ano de Publicação', fontsize=12)
plt.ylabel('Quantidade de Notícias', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.5)

# Rótulos de dados em cima de cada barra
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 1, int(yval), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## 📈 5. Gráfico Interativo com Plotly

In [ ]:
dist_df = contagem_ano.reset_index()
dist_df.columns = ["Ano", "Quantidade"]
dist_df["Ano_Str"] = dist_df["Ano"].astype(str)

fig = px.bar(
    dist_df,
    x="Ano_Str",
    y="Quantidade",
    text="Quantidade",
    title="📊 Notícias Citando a EAJ por Ano (UFRN)",
    labels={"Ano_Str": "Ano de Publicação", "Quantidade": "Número de Notícias"},
    color="Quantidade",
    color_continuous_scale="Blues"
)
fig.update_traces(textposition='outside')
fig.update_layout(template="plotly_dark", height=450)
fig.show()

## 🚀 6. Executar o Streamlit no Google Colab
Para abrir a aplicação web interativa no Colab:

In [ ]:
# Execute o comando abaixo no Colab caso queira rodar o Data App na nuvem:
# !streamlit run app.py & npx localtunnel --port 8501